# Road Damage Detection and Classification using YOLOv8
### Project-Based Learning (PBL) | Indian Road Conditions (RDD India Dataset)
**Author:** Ravish Patel  
**Hardware Environment:** Google Colab (Tesla T4 GPU, 16GB VRAM)  
**Objective:** Automate pavement distress detection across 4 categories: `D00` (Longitudinal Crack), `D10` (Transverse Crack), `D20` (Alligator Crack), and `D40` (Pothole).

## 1. Mount Google Drive & Set Workspace
Mount Google Drive where `RoadDamagePBL/` resides.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
project_dir = '/content/drive/MyDrive/RoadDamagePBL'
if os.path.exists(project_dir):
    os.chdir(project_dir)
    print(f"[✓] Current Working Directory: {os.getcwd()}")
else:
    print(f"[!] Folder {project_dir} not found. Running from current local directory.")

Mounted at /content/drive
[✓] Current Working Directory: /content/drive/MyDrive/RoadDamagePBL


## 2. Verify GPU Acceleration
Verify NVIDIA CUDA availability and GPU compute capability.

In [2]:
!nvidia-smi

Fri Nov 20 18:30:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2       |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Install Dependencies
Install Ultralytics YOLOv8 and support libraries.

In [3]:
!pip install -q ultralytics pyyaml matplotlib pandas

## 4. Dataset Preprocessing with Leakage Prevention
Convert VOC XML bounding boxes to normalized YOLO txt coordinates.
Using `--split_strategy chunk` to prevent adjacent video frames from leaking across train and val splits.

In [4]:
!python voc_to_yolo.py --base_dir . --output_dir dataset --split 0.8 --split_strategy chunk

Reading images from: ./raw_data/India/train/images
Reading XML annotations from: ./raw_data/India/train/annotations/xmls
Found 7706 annotation files.

Annotation Instance Counts:
  D00_Longitudinal_Crack   : 1734
  D10_Transverse_Crack     : 113
  D20_Alligator_Crack      : 2021
  D40_Pothole              : 3187

Split strategy 'chunk': 6164 train images, 1542 validation images.
Exporting Train split (6164 items)...
Exporting Validation split (1542 items)...

Configuration file written to: ./data.yaml
Dataset successfully staged in 'dataset/'


## 5. Inspect Generated Dataset Configuration (`data.yaml`)

In [5]:
with open('data.yaml', 'r') as f:
    print(f.read())

# YOLOv8 Dataset Configuration - Road Damage Detection (India)
path: /content/drive/MyDrive/RoadDamagePBL/dataset
train: images/train
val: images/val

nc: 4
names:
  0: D00_Longitudinal_Crack
  1: D10_Transverse_Crack
  2: D20_Alligator_Crack
  3: D40_Pothole


## 6. Train YOLOv8 Model with Transfer Learning
We train on `data.yaml` for 25 epochs using `yolov8n.pt` backbone with CIoU and DFL loss.

In [6]:
!python train.py --data data.yaml --model yolov8n.pt --epochs 25 --batch 16 --imgsz 640 --device 0

Using NVIDIA CUDA GPU: Tesla T4
Dataset config : data.yaml
Base backbone  : yolov8n.pt
Epochs         : 25
Resolution     : 640x640
Batch size     : 16
Target device  : cuda

Ultralytics 8.2.0 🚀 Python-3.10.12 torch-2.2.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
Model summary: 225 layers, 3157200 parameters, 0 gradients, 8.9 GFLOPs

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/25      3.42G      1.642      1.821      1.412         34        640: 100%|██████████| 386/386 [01:42<00:00, 3.78it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 97/97 [00:18<00:00, 5.21it/s]
                   all       1542       1468      0.384      0.241      0.218      0.104

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       5/25      3.45G      1.218      1.140      1.102         31        640: 100%|██████████| 386/386 [01:40<00:00, 3.84it/s]
                 Class     Images

## 7. Model Validation & Metric Extraction

In [7]:
!python evaluate.py --weights models/best.pt --data data.yaml --device 0

Loading checkpoint : models/best.pt
Dataset config     : data.yaml
Evaluation split   : val
Resolution         : 640x640

--- Quantitative Evaluation Metrics ---
mAP@0.50        : 0.5152
mAP@0.50-0.95   : 0.2924
Mean Precision  : 0.6041
Mean Recall     : 0.5150

Per-Class mAP@0.50 Breakdown:
  D00_Longitudinal_Crack   : 0.5182
  D10_Transverse_Crack     : 0.3421
  D20_Alligator_Crack      : 0.5514
  D40_Pothole              : 0.6483

Detailed evaluation outputs and PR curves stored in 'results/evaluation/'


## 8. Run CLI Inference on Demo Samples with Automated Severity Triage

In [8]:
!python predict.py --source demo_samples --weights models/best.pt --conf 0.25

Loading weights from: models/best.pt
Running inference on: demo_samples (conf >= 0.25)

--- Detection Summary ---

India_000005.jpg:
  - Pothole / Rutting: 1
  Condition: HIGH - Immediate repair required (pothole or severe fatigue)

India_000010.jpg:
  - Longitudinal Crack: 1
  Condition: LOW - Minor surface distress monitoring

India_000014.jpg:
  - Alligator (Fatigue) Crack: 2
  Condition: HIGH - Immediate repair required (pothole or severe fatigue)

India_000115.jpg:
  - Transverse Crack: 1
  Condition: LOW - Minor surface distress monitoring

Saved labeled detection images to: results/predictions
